# K-Nearest Neighbors and Distances

Sections follow doc/Description.md:
1. Distance metrics (Euclidean, Manhattan, Minkowski, cosine)
2. The worked example from Math/math.md
3. KNN classifier (vote) and regressor (average)
4. Scaling matters
5. Distance-weighted KNN
6. Choosing k
7. Curse of dimensionality

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(0)

## 1. Distance metrics
All functions take a query `q` (d,) and a matrix `X` (n, d) and return n distances.
- Euclidean: sqrt(sum (a-b)^2)
- Manhattan: sum |a-b|
- Minkowski: (sum |a-b|^p)^(1/p)  (p=1 Manhattan, p=2 Euclidean)
- Cosine distance: 1 - a.b / (|a||b|)

In [ ]:
def euclidean(q, X):
    return np.sqrt(((X - q) ** 2).sum(axis=1))

def manhattan(q, X):
    return np.abs(X - q).sum(axis=1)

def minkowski(q, X, p=3):
    return (np.abs(X - q) ** p).sum(axis=1) ** (1.0 / p)

def cosine(q, X):
    num = X @ q
    den = np.linalg.norm(X, axis=1) * np.linalg.norm(q) + 1e-12
    return 1.0 - num / den

a = np.array([1.0, 2.0])
B = np.array([[4.0, 6.0], [2.0, 4.0], [-1.0, -2.0]])
print("euclid   ", euclidean(a, B))
print("manhattan", manhattan(a, B))
print("mink p=3 ", minkowski(a, B, 3))
print("cosine   ", cosine(a, B), "(point 2 is parallel -> 0, point 3 opposite -> 2)")
assert np.allclose(minkowski(a, B, 2), euclidean(a, B))
assert np.allclose(minkowski(a, B, 1), manhattan(a, B))

## 2. Worked example (Math/math.md)
Apples vs oranges, Q = (160, 6), k = 3. Expected: neighbours P2, P1, P3 -> Apple.

In [ ]:
P = np.array([[150, 7], [170, 6], [140, 8], [130, 3], [120, 2], [200, 1]], float)
labels = np.array(['A', 'A', 'A', 'O', 'O', 'O'])
Q = np.array([160.0, 6.0])
d = euclidean(Q, P)
order = np.argsort(d)
for i in order:
    print(f"P{i+1}: d={d[i]:.2f} label={labels[i]}")
top = order[:3]
votes = {c: int((labels[top] == c).sum()) for c in 'AO'}
print("k=3 votes:", votes, "->", max(votes, key=votes.get))

## 3. KNN classifier and regressor
"Training" just stores the data. Prediction: compute distances, take the k smallest,
then vote (classification) or average (regression). Optional distance weighting 1/(d+eps).

In [ ]:
class KNN:
    def __init__(self, k=5, metric=euclidean, weighted=False, task='classification'):
        self.k, self.metric, self.weighted, self.task = k, metric, weighted, task

    def fit(self, X, y):
        self.X, self.y = np.asarray(X, float), np.asarray(y)
        if self.task == 'classification':
            self.classes = np.unique(self.y)
        return self

    def predict(self, Xq):
        out = []
        for q in np.asarray(Xq, float):
            d = self.metric(q, self.X)
            idx = np.argpartition(d, min(self.k, len(d) - 1))[:self.k]
            w = 1.0 / (d[idx] + 1e-9) if self.weighted else np.ones(len(idx))
            if self.task == 'classification':
                scores = np.array([w[self.y[idx] == c].sum() for c in self.classes])
                out.append(self.classes[scores.argmax()])
            else:
                out.append((w * self.y[idx]).sum() / w.sum())
        return np.array(out)

def make_blobs(n, centers, std, rng):
    X = np.vstack([rng.normal(c, std, size=(n, 2)) for c in centers])
    y = np.repeat(np.arange(len(centers)), n)
    return X, y

def split(X, y, frac, rng):
    p = rng.permutation(len(X)); m = int(frac * len(X))
    return X[p[:m]], y[p[:m]], X[p[m:]], y[p[m:]]

centers = [(0, 0), (2.5, 0.5), (1.2, 2.5)]
X, y = make_blobs(100, centers, 1.0, rng)
Xtr, ytr, Xte, yte = split(X, y, 0.7, rng)
clf = KNN(k=5).fit(Xtr, ytr)
print("test accuracy k=5:", (clf.predict(Xte) == yte).mean())

In [ ]:
# Decision regions for different k
def plot_regions(ax, model, X, y, title):
    g0, g1 = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 120),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 120))
    Z = model.predict(np.c_[g0.ravel(), g1.ravel()]).reshape(g0.shape)
    ax.contourf(g0, g1, Z, alpha=0.25, cmap='viridis', levels=np.arange(-0.5, 3))
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=12, edgecolor='k', linewidth=0.3)
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, k in zip(axes, [1, 7, 50]):
    plot_regions(ax, KNN(k=k).fit(Xtr, ytr), Xtr, ytr, f"k={k}")
plt.tight_layout(); plt.savefig('plots/01_decision_regions_k.png', dpi=110); plt.close()

## 4. Metrics compared and the scale trap
Feature 2 is multiplied by 1000 (like "income in dollars" vs "age"). Without
standardization the informative feature 1 is drowned out.

In [ ]:
rng = np.random.default_rng(1)
n = 150
y_s = rng.integers(0, 2, n)
f1 = y_s * 2.0 + rng.normal(0, 0.8, n)        # informative
f2 = rng.normal(0, 1.0, n) * 1000             # pure noise, huge scale
Xs = np.c_[f1, f2]
Xa, ya, Xb, yb = split(Xs, y_s, 0.7, rng)
mu, sd = Xa.mean(0), Xa.std(0)
acc_raw = (KNN(k=5).fit(Xa, ya).predict(Xb) == yb).mean()
acc_std = (KNN(k=5).fit((Xa - mu) / sd, ya).predict((Xb - mu) / sd) == yb).mean()
print(f"raw features: {acc_raw:.3f}   standardized: {acc_std:.3f}")

# Compare metrics on the blob data
for name, m in [('euclidean', euclidean), ('manhattan', manhattan),
                ('minkowski p=3', lambda q, X: minkowski(q, X, 3)), ('cosine', cosine)]:
    acc = (KNN(k=5, metric=m).fit(Xtr, ytr).predict(Xte) == yte).mean()
    print(f"{name:14s} acc = {acc:.3f}")

# Unit balls of each metric
th = np.linspace(0, 2 * np.pi, 400)
U = np.c_[np.cos(th), np.sin(th)]
fig, ax = plt.subplots(figsize=(5, 5))
for p, col in [(1, 'C0'), (2, 'C1'), (4, 'C2')]:
    r = (np.abs(U) ** p).sum(1) ** (1 / p)
    V = U / r[:, None]
    ax.plot(V[:, 0], V[:, 1], col, label=f'p={p}')
ax.set_aspect('equal'); ax.legend(); ax.set_title('Minkowski unit balls')
plt.tight_layout(); plt.savefig('plots/02_minkowski_unit_balls.png', dpi=110); plt.close()

## 5. Distance-weighted KNN
Closer neighbours get more say (weight 1/d). Helps with large k and uneven density.

In [ ]:
rng = np.random.default_rng(2)
X2, y2 = make_blobs(120, [(0, 0), (1.8, 1.0)], 1.0, rng)
X2tr, y2tr, X2te, y2te = split(X2, y2, 0.6, rng)
ks = [1, 3, 5, 9, 15, 25, 45, 75]
acc_u = [(KNN(k=k).fit(X2tr, y2tr).predict(X2te) == y2te).mean() for k in ks]
acc_w = [(KNN(k=k, weighted=True).fit(X2tr, y2tr).predict(X2te) == y2te).mean() for k in ks]
for k, u, w in zip(ks, acc_u, acc_w):
    print(f"k={k:3d} uniform={u:.3f} weighted={w:.3f}")
plt.figure(figsize=(6, 4))
plt.plot(ks, acc_u, 'o-', label='uniform'); plt.plot(ks, acc_w, 's-', label='distance-weighted')
plt.xlabel('k'); plt.ylabel('test accuracy'); plt.legend(); plt.title('Weighted vs uniform voting')
plt.tight_layout(); plt.savefig('plots/03_weighted_knn.png', dpi=110); plt.close()

## 6. Choosing k (with 5-fold cross-validation) and KNN regression
Small k: low bias / high variance (jagged). Large k: high bias (over-smooth).

In [ ]:
def cv_accuracy(X, y, k, folds=5, seed=0):
    idx = np.random.default_rng(seed).permutation(len(X))
    parts = np.array_split(idx, folds)
    accs = []
    for i in range(folds):
        te = parts[i]; tr = np.concatenate([parts[j] for j in range(folds) if j != i])
        accs.append((KNN(k=k).fit(X[tr], y[tr]).predict(X[te]) == y[te]).mean())
    return np.mean(accs)

rng = np.random.default_rng(3)
Xc, yc = make_blobs(100, [(0, 0), (1.5, 1.0), (0.5, 2.0)], 1.0, rng)
ks = list(range(1, 60, 2))
cv = [cv_accuracy(Xc, yc, k) for k in ks]
tr_acc = [(KNN(k=k).fit(Xc, yc).predict(Xc) == yc).mean() for k in ks]
best = ks[int(np.argmax(cv))]
print("best k by CV:", best, "acc", max(cv))
plt.figure(figsize=(6, 4))
plt.plot(ks, tr_acc, label='train'); plt.plot(ks, cv, label='5-fold CV')
plt.axvline(best, color='gray', ls='--'); plt.xlabel('k'); plt.ylabel('accuracy'); plt.legend()
plt.title('Choosing k')
plt.tight_layout(); plt.savefig('plots/04_choosing_k.png', dpi=110); plt.close()

# Regression
xr = np.sort(rng.uniform(0, 6, 80)); yr = np.sin(xr) + rng.normal(0, 0.25, 80)
grid = np.linspace(0, 6, 300)[:, None]
plt.figure(figsize=(7, 4)); plt.scatter(xr, yr, s=12, c='k')
for k in [1, 10, 40]:
    plt.plot(grid[:, 0], KNN(k=k, task='regression').fit(xr[:, None], yr).predict(grid), label=f'k={k}')
plt.legend(); plt.title('KNN regression')
plt.tight_layout(); plt.savefig('plots/05_knn_regression.png', dpi=110); plt.close()

## 7. Curse of dimensionality
In high dimension, nearest and farthest neighbours become almost equally far away,
so "nearest" loses meaning.

In [ ]:
rng = np.random.default_rng(4)
dims = [1, 2, 5, 10, 20, 50, 100, 500, 1000]
ratio = []
for dd in dims:
    Xd = rng.uniform(size=(500, dd)); q = rng.uniform(size=dd)
    dist = euclidean(q, Xd)
    ratio.append(dist.min() / dist.max())
print({dd: round(r, 3) for dd, r in zip(dims, ratio)})
plt.figure(figsize=(6, 4)); plt.semilogx(dims, ratio, 'o-')
plt.xlabel('dimension'); plt.ylabel('nearest / farthest distance')
plt.title('Distance concentration')
plt.tight_layout(); plt.savefig('plots/06_curse_of_dimensionality.png', dpi=110); plt.close()
print("done")